# 04 · Evaluation (CRISP-DM step 5)

The test set (about 1,600 images, after notebook 02's exclusions) is used **once**, here, to measure the final models fairly.

We answer the four data mining questions from the proposal:
1. Does MobileNetV2 beat the paper's 81.31% test accuracy?
2. How well does it catch Defect beans (Defect recall)?
3. Which classes get confused?
4. How well does it work on real phone photos?

In [ ]:
# --- SETUP: run this cell first -----------------------------------------
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Change this if you uploaded the project folder somewhere else in your Drive:
    PROJECT = Path("/content/drive/MyDrive/kapeklasipika")
    %pip install -q imagehash
else:
    # Running on your laptop: walk up from this notebook until we find config/config.yaml
    PROJECT = Path.cwd()
    while not (PROJECT / "config" / "config.yaml").exists():
        PROJECT = PROJECT.parent

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

from src.config import load_config, use_local_copy
cfg = load_config()
print("Project folder:", PROJECT)

In [ ]:
# Colab only: copy the images to Colab's fast local disk (Drive is slow for 8,000 files)
if IN_COLAB:
    cfg = use_local_copy(cfg)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from src.data_loading import load_split, load_clean_split, clean_split_files, CLASS_NAMES
from src.evaluate import (predict, metrics_report, per_class_table, save_report,
                          plot_confusion_matrix, misclassified, show_image_grid, PAPER_BENCHMARK_ACC)

SIZE, BS = cfg["img_size"], cfg["batch_size"]
MODELS = cfg["paths"]["models"]; METRICS = cfg["paths"]["metrics"]; FIG = cfg["paths"]["figures"]
files = clean_split_files(cfg)          # duplicates removed (see notebook 02); expect test = 1,599
val_ds = load_clean_split(cfg, "val", files=files)
test_ds = load_clean_split(cfg, "test", files=files)

names = ["baseline_cnn", "mobilenetv2_head", "mobilenetv2_finetuned", "mobilenetv2_finetuned_long"]
names = [n for n in names if (MODELS / f"{n}.keras").exists()]   # the "long" model only exists if you ran 03b
models = {n: tf.keras.models.load_model(MODELS / f"{n}.keras") for n in names}

## 0. Pick the best model on the VALIDATION set
The winner is chosen before looking at the test set, so the test score stays an honest,
one-time measurement.

In [ ]:
val_rows = []
for name, m in models.items():
    loss, acc = m.evaluate(val_ds, verbose=0)
    val_rows.append({"model": name, "val_accuracy": round(acc, 4), "val_loss": round(loss, 4)})
val_table = pd.DataFrame(val_rows).sort_values("val_accuracy", ascending=False)
BEST = val_table.iloc[0]["model"]
(METRICS / "best_model.txt").write_text(BEST)
print("Best model on validation:", BEST)
val_table

## 1. Test results for every model

In [ ]:
results, preds = [], {}
for name, m in models.items():
    y_true, y_pred, probs = predict(m, test_ds)
    preds[name] = (y_true, y_pred, probs)
    rep = metrics_report(y_true, y_pred, name)
    save_report(rep, METRICS / f"test_report_{name}.json")
    results.append({"model": name, "accuracy": rep["accuracy"], "macro_f1": rep["macro_f1"],
                    "defect_recall": rep["per_class"]["defect"]["recall"],
                    "beats_81.31%": rep["beats_paper_benchmark"]})
results.append({"model": "Febriana et al. 2022 (ResNet-18)", "accuracy": PAPER_BENCHMARK_ACC,
                "macro_f1": None, "defect_recall": None, "beats_81.31%": None})
comparison = pd.DataFrame(results)
comparison.to_csv(METRICS / "test_comparison.csv", index=False)
comparison

## 2. Success criteria check (from the proposal)

In [ ]:
rep = metrics_report(*preds[BEST][:2], BEST)
checks = pd.DataFrame([
    {"criterion": "Test accuracy > 81.31%", "result": rep["accuracy"], "met": rep["accuracy"] > 0.8131},
    {"criterion": "Macro F1 >= 0.85", "result": rep["macro_f1"], "met": rep["macro_f1"] >= 0.85},
    {"criterion": "Defect recall >= 0.90", "result": rep["per_class"]["defect"]["recall"],
     "met": rep["per_class"]["defect"]["recall"] >= 0.90},
])
display(checks)
per_class_table(rep)

## 3. Confusion matrices
Rows are the true class, columns the prediction. Off-diagonal cells are mistakes.

In [ ]:
for name in names:
    y_true, y_pred, _ = preds[name]
    plot_confusion_matrix(y_true, y_pred, f"{name} · test set", FIG / f"04_confusion_{name}.png")
    plt.show()
y_true, y_pred, _ = preds[BEST]
plot_confusion_matrix(y_true, y_pred, f"{BEST} · test set (row %)", FIG / f"04_confusion_{BEST}_pct.png", normalize=True)
plt.show()

## 4. Weak spots: the most confident mistakes
Look for patterns: lighting, blur, unusual shapes, or images that may be mislabeled.

In [ ]:
y_true, y_pred, probs = preds[BEST]
wrong = misclassified(y_true, y_pred, probs, test_ds.file_paths)
wrong.to_csv(METRICS / f"misclassified_{BEST}.csv", index=False)
print(f"{len(wrong)} mistakes out of {len(y_true)} test images")
display(wrong.groupby(["true", "predicted"]).size().sort_values(ascending=False).rename("count"))
top = wrong.head(12)
if len(top):
    show_image_grid(top["file_path"].tolist(),
                    [f"true {t} / pred {p} ({c:.0%})" for t, p, c in zip(top["true"], top["predicted"], top["confidence"])],
                    ncols=6, save_path=FIG / "04_misclassified_examples.png")
    plt.show()

## 5. Real-world test: your own phone photos
Put photos in `02_data/external/<class>/` (one bean per photo, centered, plain background).
These photos were never used for training. A drop compared with the test set is expected:
it measures how well the model handles real phone conditions.

In [ ]:
ext = cfg["paths"]["real_world_test"]
has_photos = ext.exists() and any(p.is_file() for c in CLASS_NAMES for p in (ext / c).glob("*") if (ext / c).exists())
if has_photos:
    real_ds = load_split(ext, SIZE, BS)
    yt, yp, pr = predict(models[BEST], real_ds)
    real = metrics_report(yt, yp, f"{BEST} (real-world)")
    save_report(real, METRICS / "real_world_report.json")
    print(f"Real-world accuracy: {real['accuracy']:.2%} on {len(yt)} photos (test set: {rep['accuracy']:.2%})")
    display(per_class_table(real))
    plot_confusion_matrix(yt, yp, f"{BEST} · real-world photos", FIG / "04_confusion_real_world.png"); plt.show()
else:
    print("No photos in 02_data/external yet. Add them and rerun this cell.")

## 6. Overfitting check
Compare the last training and validation accuracy from the epoch logs (notebook 03).

In [ ]:
for name, log in [("baseline_cnn", "history_baseline.csv"), ("mobilenetv2", "history_mobilenetv2.csv")]:
    h = pd.read_csv(METRICS / log)
    best = h.loc[h["val_loss"].idxmin()]
    print(f"{name}: best epoch train acc {best['accuracy']:.3f}, val acc {best['val_accuracy']:.3f}, gap {best['accuracy'] - best['val_accuracy']:.3f}")

## What these results do and do NOT prove
- They show how the model performs on USK-Coffee images (one camera setup, Indonesian Arabica beans).
- The real-world test shows how much of that carries over to phone photos of local beans,
  but with only ~50–100 photos the estimate is rough.
- They do not prove the app can replace trained human sorters; it is a decision-support tool.

Paste the outputs to Claude to draft `06_evaluation/validation_report.md` and the paper's Results section.